In [ ]:
# Example 1: a virtual network with two subnets, and the management client
# Nothing here contacts Azure: building the client and the models is local.
import ipaddress
from azure.mgmt.network import NetworkManagementClient
from azure.mgmt.network.models import AddressSpace, Subnet, VirtualNetwork


def rest_body(model):
    """The JSON body the SDK sends for a model."""
    # Newer SDK models are mappings whose as_dict() is the REST body; msrest-based models use serialize().
    serialize = getattr(model, "serialize", None)
    return serialize() if serialize is not None else model.as_dict()


class FakeCredential:
    def get_token(self, *scopes, **kwargs):
        raise AssertionError("no request should need a token in this notebook")


SUBSCRIPTION = "00000000-0000-0000-0000-000000000000"
GROUP_ID = f"/subscriptions/{SUBSCRIPTION}/resourceGroups/example-rg/providers/Microsoft.Network"

client = NetworkManagementClient(FakeCredential(), subscription_id=SUBSCRIPTION)
for group in ("virtual_networks", "subnets", "network_security_groups", "load_balancers",
              "public_ip_addresses", "network_interfaces", "application_gateways"):
    assert callable(getattr(getattr(client, group), "begin_create_or_update", None)), group
client.close()

vnet = VirtualNetwork(
    location="eastus",
    address_space=AddressSpace(address_prefixes=["10.0.0.0/16"]),
    subnets=[Subnet(name="web", address_prefix="10.0.1.0/24"),
             Subnet(name="data", address_prefix="10.0.2.0/24")],
)
space = ipaddress.ip_network(vnet.address_space.address_prefixes[0])
assert all(ipaddress.ip_network(s.address_prefix).subnet_of(space) for s in vnet.subnets)
body = rest_body(vnet)
assert body["location"] == "eastus", body
assert body["properties"]["addressSpace"] == {"addressPrefixes": ["10.0.0.0/16"]}, body
assert [(s["name"], s["properties"]["addressPrefix"]) for s in body["properties"]["subnets"]] == [
    ("web", "10.0.1.0/24"), ("data", "10.0.2.0/24")], body
print("virtual network body:", body)

In [ ]:
# Example 2: a network security group that allows HTTPS in and denies everything else
from azure.mgmt.network.models import (
    NetworkSecurityGroup, SecurityRule, SecurityRuleAccess, SecurityRuleDirection, SecurityRuleProtocol,
)

allow_https = SecurityRule(
    name="allow-https", priority=1000,
    direction=SecurityRuleDirection.INBOUND, access=SecurityRuleAccess.ALLOW,
    protocol=SecurityRuleProtocol.TCP,
    source_address_prefix="Internet", source_port_range="*",
    destination_address_prefix="10.0.1.0/24", destination_port_range="443",
)
deny_all = SecurityRule(
    name="deny-all-inbound", priority=4096,
    direction=SecurityRuleDirection.INBOUND, access=SecurityRuleAccess.DENY,
    protocol=SecurityRuleProtocol.ASTERISK,
    source_address_prefix="*", source_port_range="*",
    destination_address_prefix="*", destination_port_range="*",
)
nsg = NetworkSecurityGroup(location="eastus", security_rules=[deny_all, allow_https])

# Azure evaluates rules by ascending priority, so the HTTPS rule wins over the deny-all rule.
ordered = sorted(nsg.security_rules, key=lambda rule: rule.priority)
assert [rule.name for rule in ordered] == ["allow-https", "deny-all-inbound"]
body = rest_body(allow_https)
assert body["name"] == "allow-https", body
assert body["properties"] == {
    "protocol": "Tcp", "sourcePortRange": "*", "destinationPortRange": "443",
    "sourceAddressPrefix": "Internet", "destinationAddressPrefix": "10.0.1.0/24",
    "access": "Allow", "priority": 1000, "direction": "Inbound",
}, body
assert rest_body(deny_all)["properties"]["protocol"] == "*"
print("security rule body:", body)

In [ ]:
# Example 3: a load balancer with a frontend, a backend pool, a health probe and a rule
from azure.mgmt.network.models import (
    BackendAddressPool, FrontendIPConfiguration, LoadBalancer, LoadBalancingRule, LoadDistribution,
    Probe, ProbeProtocol, PublicIPAddress, SubResource, TransportProtocol,
)

LB_ID = f"{GROUP_ID}/loadBalancers/web-lb"
load_balancer = LoadBalancer(
    location="eastus",
    frontend_ip_configurations=[FrontendIPConfiguration(
        name="frontend", public_ip_address=PublicIPAddress(id=f"{GROUP_ID}/publicIPAddresses/web-ip"))],
    backend_address_pools=[BackendAddressPool(name="web-pool")],
    probes=[Probe(name="http-probe", protocol=ProbeProtocol.HTTP, port=80, request_path="/health",
                  interval_in_seconds=15, number_of_probes=2)],
    load_balancing_rules=[LoadBalancingRule(
        name="http",
        frontend_ip_configuration=SubResource(id=f"{LB_ID}/frontendIPConfigurations/frontend"),
        backend_address_pool=SubResource(id=f"{LB_ID}/backendAddressPools/web-pool"),
        probe=SubResource(id=f"{LB_ID}/probes/http-probe"),
        protocol=TransportProtocol.TCP, frontend_port=80, backend_port=8080,
        load_distribution=LoadDistribution.SOURCE_IP, idle_timeout_in_minutes=4,
    )],
)
rule = rest_body(load_balancer)["properties"]["loadBalancingRules"][0]
assert rule["name"] == "http", rule
assert rule["properties"]["protocol"] == "Tcp" and rule["properties"]["loadDistribution"] == "SourceIP", rule
assert (rule["properties"]["frontendPort"], rule["properties"]["backendPort"]) == (80, 8080), rule
assert rule["properties"]["probe"] == {"id": f"{LB_ID}/probes/http-probe"}, rule
probe = rest_body(load_balancer)["properties"]["probes"][0]["properties"]
assert probe == {"protocol": "Http", "port": 80, "intervalInSeconds": 15, "numberOfProbes": 2,
                 "requestPath": "/health"}, probe
print("load-balancing rule:", rule)

In [ ]:
# Example 4: a static public IP and a network interface that uses it
# The addresses come from the documentation ranges (RFC 5737 and RFC 1918).
from azure.mgmt.network.models import (
    IPAllocationMethod, NetworkInterface, NetworkInterfaceIPConfiguration, PublicIPAddress,
    PublicIPAddressSku, PublicIPAddressSkuName,
)

public_ip = PublicIPAddress(
    location="eastus",
    sku=PublicIPAddressSku(name=PublicIPAddressSkuName.STANDARD),
    public_ip_allocation_method=IPAllocationMethod.STATIC,
    ip_address="203.0.113.10",
)
body = rest_body(public_ip)
assert body["sku"] == {"name": "Standard"}, body
assert body["properties"] == {"publicIPAllocationMethod": "Static", "ipAddress": "203.0.113.10"}, body

nic = NetworkInterface(
    location="eastus",
    ip_configurations=[NetworkInterfaceIPConfiguration(
        name="ipconfig1", primary=True,
        private_ip_allocation_method=IPAllocationMethod.STATIC, private_ip_address="10.0.1.4",
        subnet=Subnet(id=f"{GROUP_ID}/virtualNetworks/example-vnet/subnets/web"),
        public_ip_address=PublicIPAddress(id=f"{GROUP_ID}/publicIPAddresses/web-ip"),
    )],
)
ip_configuration = rest_body(nic)["properties"]["ipConfigurations"][0]["properties"]
assert ip_configuration["privateIPAddress"] == "10.0.1.4", ip_configuration
assert ip_configuration["privateIPAllocationMethod"] == "Static" and ip_configuration["primary"] is True
assert ip_configuration["subnet"]["id"].endswith("/subnets/web"), ip_configuration
assert ip_configuration["publicIPAddress"]["id"].endswith("/publicIPAddresses/web-ip"), ip_configuration
assert ipaddress.ip_address("10.0.1.4") in ipaddress.ip_network("10.0.1.0/24")
print("NIC IP configuration:", ip_configuration)

In [ ]:
# Example 5: an application gateway with a WAF policy in prevention mode
from azure.mgmt.network.models import (
    ApplicationGateway, ApplicationGatewayBackendAddress, ApplicationGatewayBackendAddressPool,
    ApplicationGatewayFirewallMode, ApplicationGatewayFrontendIPConfiguration,
    ApplicationGatewayFrontendPort, ApplicationGatewayHttpListener, ApplicationGatewayProtocol,
    ApplicationGatewaySku, ApplicationGatewaySkuName, ApplicationGatewayTier,
    ApplicationGatewayWebApplicationFirewallConfiguration,
)

AGW_ID = f"{GROUP_ID}/applicationGateways/web-gateway"
gateway = ApplicationGateway(
    location="eastus",
    sku=ApplicationGatewaySku(name=ApplicationGatewaySkuName.WAF_V2, tier=ApplicationGatewayTier.WAF_V2, capacity=2),
    frontend_ip_configurations=[ApplicationGatewayFrontendIPConfiguration(
        name="public-frontend", public_ip_address=SubResource(id=f"{GROUP_ID}/publicIPAddresses/gateway-ip"))],
    frontend_ports=[ApplicationGatewayFrontendPort(name="https", port=443)],
    backend_address_pools=[ApplicationGatewayBackendAddressPool(
        name="web-backends",
        backend_addresses=[ApplicationGatewayBackendAddress(fqdn="www.example.com"),
                           ApplicationGatewayBackendAddress(ip_address="198.51.100.44")])],
    http_listeners=[ApplicationGatewayHttpListener(
        name="https-listener", protocol=ApplicationGatewayProtocol.HTTPS, host_name="www.example.com",
        frontend_ip_configuration=SubResource(id=f"{AGW_ID}/frontendIPConfigurations/public-frontend"),
        frontend_port=SubResource(id=f"{AGW_ID}/frontendPorts/https"))],
    web_application_firewall_configuration=ApplicationGatewayWebApplicationFirewallConfiguration(
        enabled=True, firewall_mode=ApplicationGatewayFirewallMode.PREVENTION,
        rule_set_type="OWASP", rule_set_version="3.2"),
)
properties = rest_body(gateway)["properties"]
assert properties["sku"] == {"name": "WAF_v2", "tier": "WAF_v2", "capacity": 2}, properties["sku"]
assert properties["frontendPorts"][0]["properties"] == {"port": 443}
backends = properties["backendAddressPools"][0]["properties"]["backendAddresses"]
assert backends == [{"fqdn": "www.example.com"}, {"ipAddress": "198.51.100.44"}], backends
listener = properties["httpListeners"][0]["properties"]
assert listener["protocol"] == "Https" and listener["hostName"] == "www.example.com", listener
assert listener["frontendPort"]["id"].endswith("/frontendPorts/https"), listener
waf = properties["webApplicationFirewallConfiguration"]
assert waf == {"enabled": True, "firewallMode": "Prevention", "ruleSetType": "OWASP", "ruleSetVersion": "3.2"}, waf
print("WAF configuration:", waf)